# 26. 아미노산 변이 패턴으로 암종 구분하기 (H26, 개인 실험)

## 배경
- 22b: 미스센스의 **"무엇이 무엇으로"** 바뀌었는지(R→H 등) 비율 → CV 0.4487 (기준 0.4125 대비 +0.036)
- 이번 실험: **"어떤 아미노산이 어떤 방식으로"** 망가졌는지를 셈
  - 원래 아미노산 20종류 × 변이 유형 5종류(침묵·미스센스·넌센스·프레임시프트·in-frame indel) = **100개**
  - 예: `K600fs` → K × 프레임시프트, `R213*` → R × 넌센스, `R132H` → R × 미스센스
  - 바뀐 아미노산이 아니라 **원래 아미노산** 기준: 프레임시프트·indel은 바뀐 아미노산이 없어서 5가지 유형을 같은 기준으로 보려면 원래 아미노산이어야 함
- 기대하는 신호
  - DNA 복구 고장 암(MSI: COAD, STAD, UCEC)은 같은 염기가 반복되는 자리에서 프레임시프트가 잘 생김 (AAA → 라이신 K, TTT → 페닐알라닌 F, CCC → 프롤린 P) → "K × 프레임시프트" 같은 열
  - 22b와 달리 침묵·넌센스·프레임시프트까지 포함

## 근거 자료 (아이디어는 이전에 본 자료에서 착안, 아래 논문들이 이 방향을 뒷받침)
- **아미노산 단위 변이 패턴은 암종마다 다르다**: Szpiech ZA 외, *Prominent features of the amino acid mutation landscape in cancer*, PLoS ONE 12(8):e0183273 (2017)
  - 29개 암종의 아미노산 치환을 NMF·PCA로 분석 → 아미노산 변이 시그니처 6개 (COSMIC 독립 데이터에서 5개 재현)
  - 어떤 암은 R→H, 어떤 암은 E→K가 주를 이룸
- **원래 아미노산에 따라 변이 빈도가 다르다**: *Arginines of the CGN codon family are Achilles' heels of cancer genes*, Scientific Reports (2024)
  - 대부분의 암종에서 가장 많이 바뀌는 원래 아미노산은 아르기닌(R) (R→H, R→Q, R→C 순)
  - 원인은 주로 CpG 자리의 C>T 변이 → 원래 아미노산은 암마다 다른 DNA 변이 과정을 간접적으로 반영
- **변이 패턴 피처만으로 암종 분류가 가능하다**: Jiao W 외, *A deep learning system accurately classifies primary and metastatic cancers using passenger mutation patterns*, Nature Communications 11:728 (2020)
  - 승객 변이 패턴으로 24개 암종 분류 정확도 약 91%, 드라이버 변이 정보를 더하면 오히려 감소
- 외부 데이터 금지 규칙에 따라 논문은 아이디어 근거로만 사용, 피처는 대회 데이터의 변이 토큰에서만 계산

## 가설
- H26a: 100개 **개수**를 더하면 CV가 오른다 (요청한 방식)
- H26b: 개수 대신 환자 전체 변이 수로 나눈 **비율**이 test 변이 수 증가(2배)에 덜 흔들려 더 안정적이다 (11/11 회의 방향)
- H26c: 22b(현재 최고 피처)에 100개 비율을 더하면 22b보다 오른다

## 검증 방법
- 모델: 10 최적 설정 XGB 고정, 데이터: `load_data()` (정규화)
- 비교 (같은 fold)
  - 기준 `any + stats` vs 26a `+ 100개 개수` vs 26b `+ 100개 비율`
  - 22b `any + ratio + 치환 비율 상위 100` vs 26c `22b + 100개 비율`
- 한 환자 안에서만 계산 → Data Leakage 아님, 100개 목록은 도메인 지식으로 고정 (fold 안에서 고를 필요 없음)
- in-frame indel은 train에 1개뿐이라 indel 20개 열은 train에서 사실상 0 (트리가 쓰지 못함, 22에서 확인한 train/test 차이)
- 채택 기준: 비교 대상보다 5 fold 중 4개 이상 상승
- 실행 환경: macOS M1 (XGB 점수는 데스크탑과 ±0.004 차이 가능), 제출 파일은 Windows에서만 저장

#### 준비: 데이터, 기본 피처, 22b 피처, 10 최적 설정

In [1]:
import sys; sys.path.append('..')
import re, time, platform, numpy as np, pandas as pd, xgboost as xgb
from collections import Counter
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import make_pipeline
from sklearn.utils.class_weight import compute_sample_weight
from src.config import SEED, WT
from src.data import load_data, get_gene_cols
from src.features import mutation_features, mutation_type, aa_substitution
from src.validation import seed_everything, encode_target, get_folds, run_cv, save_submission

print(f'{platform.system()} | CPU {platform.machine()} | xgboost {xgb.__version__}')
seed_everything()
train, test, submission = load_data()
genes = get_gene_cols(train)
y, le = encode_target(train)
folds = get_folds(y)
F_tr, F_te = mutation_features(train, genes), mutation_features(test, genes)

class BalancedXGB(xgb.XGBClassifier):
    def fit(self, X, y, **kw):
        return super().fit(X, y, sample_weight=compute_sample_weight('balanced', y), **kw)

BEST = dict(max_depth=3, learning_rate=0.1, n_estimators=200, colsample_bytree=0.3,
            tree_method='hist', eval_metric='mlogloss', random_state=SEED, n_jobs=-1)

# 22b 피처 (22 노트북과 동일): 미스센스 치환 종류 비율, 상위 100개는 fold 학습 부분에서만
def sub_ratio_matrix(df):
    rows = []
    for row in df[genes].to_numpy():
        subs = (aa_substitution(t) for v in row if v != WT for t in v.split() if mutation_type(t) == 'missense')
        c = Counter(s for s in subs if s is not None)
        tot = sum(c.values())
        rows.append({k: n / tot for k, n in c.items()} if tot else {})
    M = pd.DataFrame(rows, index=df.index).fillna(0).astype(np.float32)
    M.columns = ['sub__' + k.replace('>', '_') for k in M.columns]
    return M

class TopNSubs(BaseEstimator, TransformerMixin):
    def __init__(self, n=100):
        self.n = n
    def fit(self, X, y=None):
        sub = [c for c in X.columns if c.startswith('sub__')]
        keep = set(X[sub].sum().sort_values(ascending=False).index[:self.n])
        self.drop_ = [c for c in sub if c not in keep]
        return self
    def transform(self, X):
        return X.drop(columns=self.drop_)

S_tr = sub_ratio_matrix(train)
S_te = sub_ratio_matrix(test).reindex(columns=S_tr.columns, fill_value=0)

Darwin | CPU arm64 | xgboost 3.4.1


#### 원래 아미노산 × 변이 유형 행렬 (100개)
- `ref_aa()`: 토큰 맨 앞의 원래 아미노산 (예: `R132H` → R, `K600fs` → K, `A630_P649del` → A, `1529_1530AL>AL` → A)
  - 위치만 있는 토큰(`277del`)이나 종결코돈(`*261*`)은 20종류에 없어서 제외
- 값: 환자별 해당 칸의 변이 **개수** (26a) / 개수를 환자의 전체 변이 수(100칸 합)로 나눈 **비율** (26b, 26c)

In [2]:
AA20 = 'ACDEFGHIKLMNPQRSTVWY'
TYPES5 = ['silent', 'missense', 'nonsense', 'frameshift', 'inframe_indel']
AT_COLS = [f'at__{a}_{t}' for a in AA20 for t in TYPES5]
_LEAD = re.compile(r'^([A-Z*]+)\d')          # R132H, K600fs, WQ288fs, P233del, A630_P649del
_BLOCK = re.compile(r'^\d+_\d+([A-Z*]+)>')   # 1529_1530AL>AL

def ref_aa(token):
    m = _LEAD.match(token) or _BLOCK.match(token)
    r = m.group(1)[0] if m else None
    return r if r is not None and r in AA20 else None

def aa_type_counts(df):
    # 환자별 (원래 아미노산, 변이 유형) 개수 (한 환자 안에서만 계산 → 누수 아님)
    rows = []
    for row in df[genes].to_numpy():
        c = Counter()
        for v in row:
            if v == WT:
                continue
            for t in v.split():
                a, ty = ref_aa(t), mutation_type(t)
                if a is not None and ty in TYPES5:
                    c[f'at__{a}_{ty}'] += 1
        rows.append(c)
    return pd.DataFrame(rows, index=df.index).reindex(columns=AT_COLS, fill_value=0).fillna(0).astype(np.float32)

A_cnt_tr, A_cnt_te = aa_type_counts(train), aa_type_counts(test)
to_ratio = lambda A: A.div(A.sum(axis=1).clip(lower=1), axis=0).astype(np.float32)
A_rat_tr, A_rat_te = to_ratio(A_cnt_tr), to_ratio(A_cnt_te)
print('열 수:', A_cnt_tr.shape[1], '| train에서 한 번이라도 나온 칸:', int((A_cnt_tr.sum() > 0).sum()))
top = A_cnt_tr.sum().sort_values(ascending=False).head(8)
print('train에서 가장 많은 칸:', {k[4:]: int(v) for k, v in top.items()})

열 수: 100 | train에서 한 번이라도 나온 칸: 76
train에서 가장 많은 칸: {'R_missense': 26000, 'E_missense': 15541, 'S_missense': 13493, 'A_missense': 12649, 'P_missense': 12372, 'G_missense': 11771, 'D_missense': 10437, 'L_silent': 9156}


#### 개수 vs 비율: train vs test 평균 (관찰용)
- 개수는 test 변이가 2배라 커지고, 비율은 덜 변하는지 확인 (test 값은 피처 선택에 쓰지 않음)
- 프레임시프트 칸 예시 (MSI 관련)

In [3]:
fs_cols = [f'at__{a}_frameshift' for a in 'KFPEG']
pd.DataFrame({
    '개수 train': A_cnt_tr[fs_cols].mean(), '개수 test': A_cnt_te[fs_cols].mean(),
    '비율 train': A_rat_tr[fs_cols].mean(), '비율 test': A_rat_te[fs_cols].mean(),
}).round(4).rename(index=lambda s: s[4:]).assign(**{
    '전체 100칸 평균 (개수)': [f'train {A_cnt_tr.values.mean():.3f} / test {A_cnt_te.values.mean():.3f}'] + [''] * 4})

,개수 train,개수 test,비율 train,비율 test,전체 100칸 평균 (개수)
K_frameshift,0.1701,0.7455,0.0049,0.0051,train 0.398 / test 0.800
F_frameshift,0.0856,0.2549,0.0028,0.0024,
P_frameshift,0.1888,0.5247,0.0059,0.0049,
E_frameshift,0.0942,0.3189,0.0041,0.0043,
G_frameshift,0.1303,0.3928,0.0036,0.0037,


#### 비교: 기준 / 26a / 26b, 22b / 26c
- 시간이 오래 걸림 (XGB 5개, 약 17분)

In [4]:
cat = lambda parts: pd.concat(parts, axis=1)
SETS = {
    '기준: any + stats':            (cat([F_tr['any'], F_tr['stats']]), cat([F_te['any'], F_te['stats']]), None),
    '26a: 기준 + 100개 개수':        (cat([F_tr['any'], F_tr['stats'], A_cnt_tr]), cat([F_te['any'], F_te['stats'], A_cnt_te]), None),
    '26b: 기준 + 100개 비율':        (cat([F_tr['any'], F_tr['stats'], A_rat_tr]), cat([F_te['any'], F_te['stats'], A_rat_te]), None),
    '22b: any + ratio + 치환 100':   (cat([F_tr['any'], F_tr['ratio'], S_tr]), cat([F_te['any'], F_te['ratio'], S_te]), 100),
    '26c: 22b + 100개 비율':         (cat([F_tr['any'], F_tr['ratio'], S_tr, A_rat_tr]), cat([F_te['any'], F_te['ratio'], S_te, A_rat_te]), 100),
}
R = {}
for name, (X, X_te, n) in SETS.items():
    t = time.time()
    mk = (lambda: BalancedXGB(**BEST)) if n is None else (lambda: make_pipeline(TopNSubs(n), BalancedXGB(**BEST)))
    R[name] = run_cv(mk, X, y, X_te, folds=folds, verbose=False)
    print(f"{name}: CV {R[name]['cv_mean']:.4f} ± {R[name]['cv_std']:.4f} (OOF {R[name]['oof_score']:.4f}, {time.time() - t:.0f}s)")

fs = lambda k: np.array(R[k]['fold_scores'])
PAIR = {'26a: 기준 + 100개 개수': '기준: any + stats', '26b: 기준 + 100개 비율': '기준: any + stats',
        '26c: 22b + 100개 비율': '22b: any + ratio + 치환 100', '22b: any + ratio + 치환 100': '기준: any + stats'}
rows = []
for name in R:
    ref = PAIR.get(name)
    rows.append({'피처': name, 'CV': round(fs(name).mean(), 4), 'std': round(fs(name).std(), 4), 'OOF': round(R[name]['oof_score'], 4),
                 '비교 대상': ref or '-', '비교 대상 대비': round(fs(name).mean() - fs(ref).mean(), 4) if ref else '-',
                 '높은 fold': f'{int((fs(name) > fs(ref)).sum())}/5' if ref else '-'})
pd.DataFrame(rows)

기준: any + stats: CV 0.4125 ± 0.0109 (OOF 0.4131, 153s)
26a: 기준 + 100개 개수: CV 0.4434 ± 0.0026 (OOF 0.4439, 164s)
26b: 기준 + 100개 비율: CV 0.4385 ± 0.0105 (OOF 0.4389, 179s)
22b: any + ratio + 치환 100: CV 0.4487 ± 0.0065 (OOF 0.4497, 171s)
26c: 22b + 100개 비율: CV 0.4560 ± 0.0116 (OOF 0.4566, 186s)


,피처,CV,std,OOF,비교 대상,비교 대상 대비,높은 fold
0,기준: any + stats,0.4125,0.0109,0.4131,-,-,-
1,26a: 기준 + 100개 개수,0.4434,0.0026,0.4439,기준: any + stats,0.0309,5/5
2,26b: 기준 + 100개 비율,0.4385,0.0105,0.4389,기준: any + stats,0.0259,5/5
3,22b: any + ratio + 치환 100,0.4487,0.0065,0.4497,기준: any + stats,0.0362,5/5
4,26c: 22b + 100개 비율,0.4560,0.0116,0.4566,22b: any + ratio + 치환 100,0.0073,4/5


#### 중요도 상위 100개 칸 (해석용)
- 26b를 train 전체로 한 번 학습해 어떤 칸을 쓰는지 확인 (점수 계산용 아님)

In [5]:
X26b = SETS['26b: 기준 + 100개 비율'][0]
m = BalancedXGB(**BEST).fit(X26b, y)
imp = pd.Series(m.feature_importances_, index=X26b.columns)
top = imp[imp.index.str.startswith('at__')].sort_values(ascending=False).head(12)
pd.DataFrame({'중요도': top.round(4)}).rename(index=lambda s: s[4:])

,중요도
W_frameshift,0.0025
E_missense,0.0015
F_silent,0.0015
I_silent,0.0013
R_missense,0.0013
V_missense,0.0012
Q_missense,0.0012
G_nonsense,0.0010
C_nonsense,0.0010
L_silent,0.0010


#### test 예측 분포 (관찰용, 판단 기준 아님)

In [6]:
pd.DataFrame({k: pd.Series(le.classes_[v['test_proba'].argmax(1)]).value_counts(normalize=True)
              for k, v in R.items()}).fillna(0).sort_values('기준: any + stats', ascending=False).head(8).mul(100).round(1).astype(str) + '%'

,기준: any + stats,26a: 기준 + 100개 개수,26b: 기준 + 100개 비율,22b: any + ratio + 치환 100,26c: 22b + 100개 비율
STES,13.6%,17.6%,17.6%,21.8%,23.0%
COAD,10.9%,12.9%,12.7%,12.4%,11.3%
TGCT,5.5%,3.5%,3.4%,2.2%,1.9%
LUSC,5.1%,4.2%,4.4%,1.8%,2.2%
OV,4.9%,4.9%,4.9%,4.3%,4.5%
BRCA,4.9%,4.7%,4.8%,5.3%,5.0%
BLCA,4.4%,1.4%,1.1%,1.2%,0.9%
UCEC,3.9%,3.7%,4.2%,3.8%,3.5%


#### 제출 파일 (데스크탑에서만 저장)
- 26 설정 중 비교 대상보다 5 fold 중 4개 이상 오른 것 가운데 CV 최고를 저장

In [7]:
cands = {k: R[k] for k in R if k.startswith('26') and (fs(k) > fs(PAIR[k])).sum() >= 4}
if not cands:
    print('채택 기준(비교 대상보다 5 fold 중 4개 이상)을 넘는 26 설정 없음 → 저장 안 함')
else:
    best = max(cands, key=lambda k: cands[k]['cv_mean'])
    print('저장 대상:', best)
    if platform.system() == 'Windows':
        save_submission(submission, cands[best]['test_proba'], le, name=f'xgb_d3_cs0.3_aatype26{best[2]}', cv_score=cands[best]['cv_mean'])
    else:
        print(f'{platform.system()}에서 실행 → 제출 파일 저장 안 함 (데스크탑에서 실행하면 저장)')

저장 대상: 26c: 22b + 100개 비율
Darwin에서 실행 → 제출 파일 저장 안 함 (데스크탑에서 실행하면 저장)


## 결과 (macOS M1)

**미리 확인 (관찰용)**: 100칸 중 76칸이 train에 나타남. **개수**는 test에서 평균 약 2배(0.398 → 0.800, 예: K × 프레임시프트 0.17 → 0.75), **비율**은 거의 그대로(0.0049 → 0.0051)

| 피처 | CV (± std) | OOF | 비교 대상 | 대비 | 높은 fold |
| :--- | :-: | :-: | :--- | :-: | :-: |
| 기준: any + stats | 0.4125 ± 0.0109 | 0.4131 | - | - | - |
| **26a: 기준 + 100개 개수** | 0.4434 ± 0.0026 | 0.4439 | 기준 | +0.0309 | 5/5 |
| 26b: 기준 + 100개 비율 | 0.4385 ± 0.0105 | 0.4389 | 기준 | +0.0259 | 5/5 |
| 22b: any + ratio + 치환 100 | 0.4487 ± 0.0065 | 0.4497 | 기준 | +0.0362 | 5/5 |
| **26c: 22b + 100개 비율** | **0.4560 ± 0.0116** | **0.4566** | 22b | **+0.0073** | **4/5** |

- **요청한 방식(26a, 100개 개수)도 기준 대비 크게 오름** (+0.031, 5/5 fold, std도 가장 작음 0.0026)
- 개수(26a)가 비율(26b)보다 CV는 +0.005 높음 → 다만 개수는 test에서 2배로 커져, 11/11 회의에서 걱정한 "개수 흔들림"이 그대로 있음
- 22b에 100개 비율을 더한 26c가 **XGB 단독 중 CV 최고** (22b 대비 +0.0073, 4/5 fold → 채택 기준 통과)
  - 단, 개선 폭(0.0073)이 std(0.0116)보다 작음 → CV로는 "같은 점수"에 가까움
- 중요도 상위 칸: W × 프레임시프트, E × 미스센스, F·I·L × 침묵, G·C × 넌센스 → 미스센스뿐 아니라 침묵·넌센스·프레임시프트 칸도 사용 (22b가 못 보는 정보)

**test 예측 분포 (관찰용)**
- STES: 기준 13.6% → 26a·26b 17.6% → 22b 21.8% → 26c 23.0% (치환·아미노산 피처를 넣을수록 STES 증가)
- COAD: 10.9~12.9%로 큰 변화 없음, TGCT·LUSC·BLCA는 감소

## 결론
- **H26a 채택 (CV 기준)**: 원래 아미노산 × 변이 유형 100개 개수만으로 기준 대비 +0.031 (5/5)
- **H26b**: 비율도 +0.026 (5/5)이지만 개수보다 낮음 → CV만 보면 개수가 낫고, test 2배 문제를 생각하면 비율이 안전한 편 → 어느 쪽이 맞는지는 LB로만 판단 가능
- **H26c 채택 (기준 통과)**: 22b + 100개 비율이 XGB 단독 CV 최고 (0.4560), 다만 22b 대비 차이가 std 안
- ⚠️ 22b 계열과 마찬가지로 test 예측이 STES로 더 이동 (23.0%) → LB 확인 필요
- 다음 후보로 쓴다면: 26c XGB 0.7 + LR 0.3 (22의 후보 C 방식) → 22 후보 C(0.4557)와 비교
- 제출 파일: 데스크탑에서 실행하면 `xgb_d3_cs0.3_aatype26c` 저장 (채택 기준 통과 설정 중 CV 최고)
